# 05 · Store file cleaning and join check

`store_locations.csv` has its own problems: duplicate store ids in other formats, inconsistent state names, and a store with no coordinates. This notebook cleans it and checks that transactions join to it.

In [1]:
import os
import pandas as pd
import numpy as np

DATA_DIR = next(d for d in ('data', '../data', '../../data') if os.path.isdir(d))
MISSING_TOKENS = {'', 'null', 'n/a', 'na', 'nan', 'none'}
DECISION_LOG = []

def log_decision(step, decision, rows_touched, rows_before=None, rows_after=None):
    DECISION_LOG.append({'step': step, 'decision': decision, 'rows_touched': int(rows_touched),
                         'rows_before': rows_before, 'rows_after': rows_after})


In [2]:
CATEGORY_MAP = {
    'alcohol': 'Alcohol', 'alcoholic beverages': 'Alcohol',
    'baby': 'Baby', 'baby products': 'Baby',
    'breakfast': 'Breakfast', 'breakfast foods': 'Breakfast',
    'electronics': 'Electronics',
    'emergency': 'Emergency', 'emergency supplies': 'Emergency',
    'entertainment': 'Entertainment', 'games & entertainment': 'Entertainment',
    'grocery': 'Grocery', 'groceries': 'Grocery',
    'health': 'Health', 'health & wellness': 'Health',
    'household': 'Household', 'household essentials': 'Household',
    'snacks': 'Snacks',
}
TS_FORMATS = ['%Y-%m-%d %H:%M:%S', '%Y-%m-%dT%H:%M:%S', '%m/%d/%Y %H:%M',
              '%d-%b-%Y %H:%M', '%m-%d-%Y %H:%M:%S']

def load_transactions():
    return pd.read_csv(os.path.join(DATA_DIR, 'walmart_transactions.csv'), dtype=str)

def standardize_missing(df):
    """Strip whitespace; turn '', NULL, N/A, NaN... into real NaN."""
    out = df.copy()
    touched = 0
    for c in out.columns:
        s = out[c].str.strip()
        newly_na = s.str.lower().isin(MISSING_TOKENS) & out[c].notna()
        touched += int(newly_na.sum())
        out[c] = s.mask(newly_na)
    log_decision('missing', 'strip whitespace; blank/NULL/N-A tokens -> NaN (cells, not rows)', touched, len(df), len(out))
    return out

def remove_duplicates(df):
    n0 = len(df)
    out = df.drop_duplicates()
    log_decision('duplicates', 'drop exact duplicate rows (keep first)', n0 - len(out), n0, len(out))
    n1 = len(out)
    out = out.drop_duplicates('transaction_id')
    log_decision('duplicates', 'drop repeated transaction_id (keep first)', n1 - len(out), n1, len(out))
    return out

def clean_unit_price(df):
    out = df.copy()
    raw = out['unit_price']
    dollar = int(raw.str.contains('$', regex=False, na=False).sum())
    out['unit_price'] = pd.to_numeric(raw.str.replace('$', '', regex=False).str.replace(',', '', regex=False), errors='coerce')
    bad = int(out['unit_price'].isna().sum() - raw.isna().sum())
    log_decision('unit_price', "remove '$', cast text -> float", dollar, len(df), len(out))
    log_decision('unit_price', 'values that still failed numeric cast -> NaN', bad, len(df), len(out))
    return out

def clean_quantity(df):
    out = df.copy()
    out['quantity'] = pd.to_numeric(out['quantity'], errors='coerce')
    n0 = len(out)
    out = out[out['quantity'] > 0]
    log_decision('quantity', 'drop rows with negative/zero/unparseable quantity (entry errors; sign-flip not assumed)', n0 - len(out), n0, len(out))
    return out

def parse_timestamps(df):
    out = df.copy()
    ts = pd.Series(pd.NaT, index=out.index, dtype='datetime64[ns]')
    for fmt in TS_FORMATS:
        todo = ts.isna() & out['timestamp'].notna()
        ts[todo] = pd.to_datetime(out.loc[todo, 'timestamp'], format=fmt, errors='coerce')
    unparsed = int((ts.isna() & out['timestamp'].notna()).sum())
    missing = int(out['timestamp'].isna().sum())
    out['timestamp'] = ts
    log_decision('timestamp', f'parse {len(TS_FORMATS)} mixed formats into one datetime (naive local time); {unparsed} unparseable', int(ts.notna().sum()), len(df), len(out))
    log_decision('timestamp', 'missing timestamps kept (NaT) but excluded from any date-based analysis; not imputed', missing, len(df), len(out))
    return out

def clean_store_id(df):
    out = df.copy()
    raw = out['store_id']
    out['store_id'] = (raw.str.upper().str.replace('_', '-', regex=False)
                          .str.replace(r'^FL-?(\d{3})$', r'FL-\1', regex=True))
    log_decision('store_id', "uppercase, strip, '_'/missing '-' -> canonical 'FL-###'", int((raw != out['store_id']).sum()), len(df), len(out))
    return out

def clean_category(df):
    out = df.copy()
    raw = out['category']
    out['category'] = raw.str.lower().map(CATEGORY_MAP)
    log_decision('category', f'collapse {raw.nunique()} spellings into {out["category"].nunique()} canonical categories', int((raw != out['category']).sum()), len(df), len(out))
    return out

def clean_transactions():
    DECISION_LOG.clear()
    df = load_transactions()
    for step in (standardize_missing, remove_duplicates, clean_unit_price, clean_quantity,
                 parse_timestamps, clean_store_id, clean_category):
        df = step(df)
    df['product_name_raw'] = df['product_name']
    return df.reset_index(drop=True)


In [3]:
def clean_stores():
    s = pd.read_csv(os.path.join(DATA_DIR, 'store_locations.csv'), dtype={'zip_code': str})
    n0 = len(s)
    s['store_id'] = (s['store_id'].str.strip().str.upper().str.replace('_', '-', regex=False)
                     .str.replace(r'^FL-?(\d{3})$', r'FL-\1', regex=True))
    s = s.drop_duplicates('store_id').reset_index(drop=True)
    log_decision('stores', 'canonicalise store_id and drop duplicate store rows', n0 - len(s), n0, len(s))
    changed = int((s['state'] != 'FL').sum())
    s['state'] = 'FL'
    log_decision('stores', "normalise state ('Florida', 'Fla.') to 'FL'", changed, len(s), len(s))
    # Tallahassee has no coordinates; city-centre coordinates filled from public knowledge (approximate), needed for the weather call
    fill = {'FL-619': (30.4383, -84.2807)}
    for sid, (lat, lon) in fill.items():
        m = s['store_id'] == sid
        s.loc[m & s['latitude'].isna(), ['latitude', 'longitude']] = (lat, lon)
    log_decision('stores', 'fill missing lat/lon for Tallahassee (FL-619) with approximate city-centre coordinates', len(fill), len(s), len(s))
    return s

In [4]:
tx_clean = clean_transactions()
stores = clean_stores()
print(stores[['store_id', 'city', 'state', 'latitude', 'longitude', 'region']])
assert stores['store_id'].is_unique and stores[['latitude', 'longitude']].notna().all().all()

   store_id            city state  latitude  longitude      region
0    FL-239         Orlando    FL   28.5383   -81.3792     Central
1    FL-105           Tampa    FL   27.9506   -82.4572        West
2    FL-330           Miami    FL   25.7617   -80.1918       South
3    FL-412    Jacksonville    FL   30.3322   -81.6557       North
4    FL-287       Kissimmee    FL   28.2920   -81.4076     Central
5    FL-156   Daytona Beach    FL   29.2108   -81.0228        East
6    FL-501      Fort Myers    FL   26.6406   -81.8723        West
7    FL-078     Gainesville    FL   29.6516   -82.3248       North
8    FL-619     Tallahassee    FL   30.4383   -84.2807   Panhandle
9    FL-445  St. Petersburg    FL   27.7676   -82.6403        West
10   FL-302        Lakeland    FL   28.0395   -81.9498     Central
11   FL-710       Pensacola    FL   30.4213   -87.2169   Panhandle


In [5]:
before = len(tx_clean)
joined = tx_clean.merge(stores, on='store_id', how='left', validate='many_to_one', indicator=True)
print('rows before join:', before, '| after:', len(joined))
print(joined['_merge'].value_counts())
assert len(joined) == before and (joined['_merge'] == 'both').all(), 'some transactions did not match a store'
print('\ntransactions per store:'); print(joined.groupby(['store_id', 'city']).size())

rows before join: 25402 | after: 25402
_merge
both          25402
left_only         0
right_only        0
Name: count, dtype: int64

transactions per store:
store_id  city          
FL-078    Gainesville       2067
FL-105    Tampa             2123
FL-156    Daytona Beach     2161
FL-239    Orlando           2084
FL-287    Kissimmee         2123
FL-302    Lakeland          2179
FL-330    Miami             2140
FL-412    Jacksonville      2104
FL-445    St. Petersburg    2166
FL-501    Fort Myers        2063
FL-619    Tallahassee       2056
FL-710    Pensacola         2136
dtype: int64


In [6]:
pd.DataFrame(DECISION_LOG).tail(3)

,step,decision,rows_touched,rows_before,rows_after
10,stores,canonicalise store_id and drop duplicate store...,2,14,12
11,stores,"normalise state ('Florida', 'Fla.') to 'FL'",3,12,12
12,stores,fill missing lat/lon for Tallahassee (FL-619) ...,1,12,12
